In [1]:
import os
import glob
import pandas as pd
import numpy as np
import ee

import yaml

# ------------------------------------------------------------
# 1. Load Configuration
# ------------------------------------------------------------
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)


PROJECT_ID = config['gee']['project_id']
INPUT_DIR = config['paths']['aez_data_dir']
DOWNLOAD_DIR = config['paths']['nirv_ph_download_dir']
GDRIVE_FOLDER = config['gdrive_exports']['nirv_ph_folder']

START_DATE = config['parameters']['start_date']
END_DATE = config['parameters']['end_date']

# 1. Initialize GEE
ee.Authenticate()
ee.Initialize(project=PROJECT_ID, opt_url='https://earthengine-highvolume.googleapis.com')
print("GEE Initialized successfully.")

# 2. Configuration
# INPUT_DIR = "./shc_data/AEZS/AGRI_2023-24/"
# GDRIVE_FOLDER = "GEE_NIRV_pH_All_AEZs_ratinder"
CHUNK_SIZE = 5000 

def get_combined_features(roi):
    """Builds a single image containing 3 Seasonal NIRv bands and 2 SoilGrids pH bands."""
    
    # --- 1. Landsat Seasonal NIRv ---
    def process_landsat(image):
        qa = image.select('QA_PIXEL')
        cloud = qa.bitwiseAnd(1 << 3).eq(0)
        cloud_shad = qa.bitwiseAnd(1 << 4).eq(0)
        cirrus = qa.bitwiseAnd(1 << 2).eq(0)
        snow = qa.bitwiseAnd(1 << 5).eq(0)
        mask = cloud.And(cloud_shad).And(cirrus).And(snow)

        scaled = image.select(['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']) \
                      .multiply(0.0000275).add(-0.2) \
                      .rename(['BLUE','GREEN','RED','NIR','SWIR1','SWIR2'])

        ndvi = scaled.normalizedDifference(['NIR', 'RED']).rename('NDVI')
        nirv = ndvi.multiply(scaled.select('NIR')).rename('NIRv')
        
        # Crucial: preserve time_start for seasonal filtering
        return nirv.updateMask(mask).copyProperties(image, ['system:time_start'])

    # Load and filter Landsat Collection
    LS = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
        .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')) \
        .filterBounds(roi) \
        .filterDate('2023-07-01', '2024-06-30') \
        .map(process_landsat)

    # Seasonal Reducers
    def get_season(start, end, name):
        season_ic = LS.filterDate(start, end)
        return ee.Image(ee.Algorithms.If(
            season_ic.size().gt(0),
            season_ic.median().rename(name).unmask(0),
            ee.Image.constant(0).toFloat().rename(name)
        ))

    nirv_kharif = get_season('2023-07-01', '2023-10-31', 'NIRv_Kharif')
    nirv_rabi   = get_season('2023-11-01', '2024-03-31', 'NIRv_Rabi')
    nirv_zaid   = get_season('2024-04-01', '2024-06-30', 'NIRv_Zaid')
    
    nirv_stack = nirv_kharif.addBands([nirv_rabi, nirv_zaid])

    # --- 2. SoilGrids pH ---
    ph_asset = ee.Image('projects/soilgrids-isric/phh2o_mean') \
                 .select(['phh2o_0-5cm_mean', 'phh2o_5-15cm_mean'], ['pH_0-5', 'pH_5-15'])
    # Scale real pH values (divide by 10)
    ph_stack = ph_asset.divide(10.0)

    # Combine all 5 bands into one image
    return nirv_stack.addBands(ph_stack).toFloat()

/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


GEE Initialized successfully.


In [2]:
# Get ONLY the LULC filtered files
filtered_files = glob.glob(os.path.join(INPUT_DIR, "*_LULC_Filtered.csv"))

print(f"Found {len(filtered_files)} LULC Filtered files to process.")

for file_path in filtered_files:
    # Extract just the "AEZ_X" part of the name
    aez_name = os.path.basename(file_path).replace('_LULC_Filtered.csv', '')
    print(f"\n--- Submitting {aez_name} ---")
    
    df = pd.read_csv(file_path)
    # Ensure orig_idx exists (it should from the previous script, but just in case)
    if 'orig_idx' not in df.columns:
        df = df.reset_index().rename(columns={'index': 'orig_idx'})
    
    num_chunks = int(np.ceil(len(df) / CHUNK_SIZE))
    print(f"Submitting {num_chunks} tasks...")

    for i in range(num_chunks):
        start = i * CHUNK_SIZE
        end = min((i + 1) * CHUNK_SIZE, len(df))
        
        # Strip payload to bare minimum
        chunk_df = df.iloc[start:end][['latitude', 'longitude', 'orig_idx']]
        
        features = [
            ee.Feature(ee.Geometry.Point([row.longitude, row.latitude]), {'orig_idx': int(row.orig_idx)}) 
            for row in chunk_df.itertuples(index=False)
        ]
        
        fc = ee.FeatureCollection(features)
        
        # Get the bounding box for this specific chunk to limit image generation
        roi = fc.geometry().bounds()
        combined_image = get_combined_features(roi)
        
        # Sample the 5 bands at 30m resolution (Landsat scale)
        sampled_fc = combined_image.sampleRegions(
            collection=fc,
            scale=30,
            geometries=False
        )
        
        task_name = f'{aez_name}_NIRV_pH_Part_{i+1:02d}'
        task = ee.batch.Export.table.toDrive(
            collection=sampled_fc,
            description=task_name,
            fileFormat='CSV',
            folder=GDRIVE_FOLDER,
            fileNamePrefix=task_name
        )
        task.start()
        
    print(f"All tasks for {aez_name} submitted.")

print("\n✅ All feature extraction tasks active. Monitor at: https://code.earthengine.google.com/tasks")

Found 18 LULC Filtered files to process.

--- Submitting AEZ_11 ---
Submitting 16 tasks...
All tasks for AEZ_11 submitted.

--- Submitting AEZ_8 ---
Submitting 22 tasks...
All tasks for AEZ_8 submitted.

--- Submitting AEZ_14 ---
Submitting 4 tasks...
All tasks for AEZ_14 submitted.

--- Submitting AEZ_10 ---
Submitting 14 tasks...
All tasks for AEZ_10 submitted.

--- Submitting AEZ_9 ---
Submitting 26 tasks...
All tasks for AEZ_9 submitted.

--- Submitting AEZ_15 ---
Submitting 30 tasks...
All tasks for AEZ_15 submitted.

--- Submitting AEZ_13 ---
Submitting 21 tasks...
All tasks for AEZ_13 submitted.

--- Submitting AEZ_16 ---
Submitting 1 tasks...
All tasks for AEZ_16 submitted.

--- Submitting AEZ_12 ---
Submitting 12 tasks...
All tasks for AEZ_12 submitted.

--- Submitting AEZ_17 ---
Submitting 1 tasks...
All tasks for AEZ_17 submitted.

--- Submitting AEZ_7 ---
Submitting 3 tasks...
All tasks for AEZ_7 submitted.

--- Submitting AEZ_2 ---
Submitting 18 tasks...
All tasks for AEZ_

In [3]:
import pandas as pd
import glob
import os

# DOWNLOAD_DIR = './NIRV_pH' # Update this to the folder where you downloaded the Drive files
# INPUT_DIR = "./shc_data/AEZS/AGRI_2023-24/"

# Re-grab the filtered files
filtered_files = glob.glob(os.path.join(INPUT_DIR, "*_LULC_Filtered.csv"))

for file_path in filtered_files:
    aez_name = os.path.basename(file_path).replace('_LULC_Filtered.csv', '')
    
    # 1. Find all downloaded chunks for this specific AEZ
    chunk_files = glob.glob(os.path.join(DOWNLOAD_DIR, f"{aez_name}_NIRV_pH_Part_*.csv"))
    
    if not chunk_files:
        print(f"Skipping {aez_name}: No downloaded chunks found.")
        continue
        
    # 2. Robustly load chunks (skipping empty files)
    valid_dfs = []
    for f in chunk_files:
        try:
            if os.path.getsize(f) > 0:
                temp_df = pd.read_csv(f)
                if not temp_df.empty:
                    valid_dfs.append(temp_df)
        except pd.errors.EmptyDataError:
            pass
            
    if not valid_dfs:
        print(f"Skipping {aez_name}: All chunks were empty.")
        continue

    # Combine valid chunks
    df_features = pd.concat(valid_dfs, ignore_index=True)
    
    # Keep only the index and the 5 new bands
    cols_to_keep = ['orig_idx', 'NIRv_Kharif', 'NIRv_Rabi', 'NIRv_Zaid', 'pH_0-5', 'pH_5-15']
    # Ensure all columns exist (in case a band failed entirely)
    available_cols = [c for c in cols_to_keep if c in df_features.columns]
    df_features = df_features[available_cols]
    
    # 3. Load the LULC filtered master file
    df_master = pd.read_csv(file_path)
    if 'orig_idx' not in df_master.columns:
        df_master = df_master.reset_index().rename(columns={'index': 'orig_idx'})
    
    # 4. Merge robustly on orig_idx (Left join ensures we keep all croplands, even if GEE missed a pixel)
    df_final = pd.merge(df_master, df_features, on='orig_idx', how='left')
    
    # 5. Calculate pH_avg locally (Mean of 0-5 and 5-15, skipping NaNs)
    if 'pH_0-5' in df_final.columns and 'pH_5-15' in df_final.columns:
        df_final['pH_avg'] = df_final[['pH_0-5', 'pH_5-15']].mean(axis=1, skipna=True)
    
    # Optional: Drop orig_idx if you no longer need it for tracking
    df_final = df_final.drop(columns=['orig_idx'], errors='ignore')
    
    # 6. Save final output
    output_path = os.path.join(INPUT_DIR, f"{aez_name}.csv")
    df_final.to_csv(output_path, index=False)
    
    print(f"--- {aez_name} Finalized ---")
    print(f"Total Rows: {len(df_final)}")
    print(f"Saved to: {output_path}\n")

print("✅ All AEZs successfully merged with NIRv and pH. Dataset compilation complete!")

--- AEZ_11 Finalized ---
Total Rows: 76411
Saved to: ./experiments/AEZS/AGRI_2023-24/AEZ_11_ReadyForTraining.csv

--- AEZ_8 Finalized ---
Total Rows: 105113
Saved to: ./experiments/AEZS/AGRI_2023-24/AEZ_8_ReadyForTraining.csv

--- AEZ_14 Finalized ---
Total Rows: 16542
Saved to: ./experiments/AEZS/AGRI_2023-24/AEZ_14_ReadyForTraining.csv

--- AEZ_10 Finalized ---
Total Rows: 67872
Saved to: ./experiments/AEZS/AGRI_2023-24/AEZ_10_ReadyForTraining.csv

--- AEZ_9 Finalized ---
Total Rows: 127825
Saved to: ./experiments/AEZS/AGRI_2023-24/AEZ_9_ReadyForTraining.csv

--- AEZ_15 Finalized ---
Total Rows: 148020
Saved to: ./experiments/AEZS/AGRI_2023-24/AEZ_15_ReadyForTraining.csv

--- AEZ_13 Finalized ---
Total Rows: 100475
Saved to: ./experiments/AEZS/AGRI_2023-24/AEZ_13_ReadyForTraining.csv

--- AEZ_16 Finalized ---
Total Rows: 3403
Saved to: ./experiments/AEZS/AGRI_2023-24/AEZ_16_ReadyForTraining.csv

--- AEZ_12 Finalized ---
Total Rows: 56578
Saved to: ./experiments/AEZS/AGRI_2023-24/AEZ_